In [ ]:
print('===== NATIVE GELU BWD BITEXACT TEST =====')
print('===== NATIVE GELU V5 - AVX2 VECTOR HELPER =====')
print('===== NATIVE GELU V4 - SAME FLAGS AS NATIVE SO =====')
print('===== NATIVE GELU V3 - ARGTYPES PTR INT PTR INT LL LL =====')
print('===== NATIVE GELU FWD V2 - argtypes fixed =====')
print('===== NATIVE GELU FWD BITEXACT TEST =====')
print('===== NATIVE BIASGRAD BITEXACT V2 - 7 args =====')
print('===== NATIVE BIASGRAD BITEXACT TEST =====')
# 1/4 - clone main + prototype (BiasAddRows .tkv)
import subprocess, os
print('===== NATIVE BIASADD BITEXACT TEST =====')
for d,br in [('/content/TVmain','main'),('/content/TVcpu','prototype/cuda-backend')]:
    subprocess.run('rm -rf '+d, shell=True)
    subprocess.run('git clone --depth 1 --branch '+br+' https://github.com/nguyenhungtran18/TokenVector.git '+d, shell=True, capture_output=True, text=True, timeout=600)
    print(br, subprocess.check_output('git -C '+d+' rev-parse --short HEAD', shell=True).decode().strip())

In [ ]:
# 2/4 - build native .so (has BiasAddRows) + .tkv-generated C .so
import subprocess, os
root='/content/TVmain/Release/tools/simd_native'
srcs='tkv_simd_native.c tkv_cuda_native.c tkv_linear_dw.c tkv_bwd_ops.c tkv_flash_attn.c tkv_step124.c tkv_prof.c tkv_attn_cuda.c'
p=subprocess.run('cd '+root+' && gcc -O2 -mavx2 -mfma -shared -fPIC '+srcs+' -lm -lpthread -ldl -o tkv_simd_native.so', shell=True, capture_output=True, text=True, timeout=600)
print('native build rc=',p.returncode)
ex='/content/TVcpu/Release/cuda_backend/examples'
cc=ex+'/gelu_bwd_cpu.cpu.c'; so=ex+'/gelu_bwd_cpu.so'
# inject FULL AVX2 gelu BWD implementation (same as native header + bwd_ops)
cc_path=ex+'/gelu_bwd_cpu.cpu.c'
txt=open(cc_path).read()
avx=r"""
static inline __m256 tkv_exp_ps(__m256 x) {
    const __m256 LOG2E = _mm256_set1_ps(1.4426950408889634f);
    const __m256 LN2   = _mm256_set1_ps(0.6931471805599453f);
    __m256 kf = _mm256_round_ps(_mm256_mul_ps(x, LOG2E),
                                _MM_FROUND_TO_NEAREST_INT | _MM_FROUND_NO_EXC);
    __m256 r  = _mm256_fnmadd_ps(kf, LN2, x);
    __m256 p  = _mm256_set1_ps(2.48015873e-05f);
    p = _mm256_fmadd_ps(p, r, _mm256_set1_ps(1.98412698e-04f));
    p = _mm256_fmadd_ps(p, r, _mm256_set1_ps(1.38888889e-03f));
    p = _mm256_fmadd_ps(p, r, _mm256_set1_ps(8.33333333e-03f));
    p = _mm256_fmadd_ps(p, r, _mm256_set1_ps(4.16666667e-02f));
    p = _mm256_fmadd_ps(p, r, _mm256_set1_ps(1.66666667e-01f));
    p = _mm256_fmadd_ps(p, r, _mm256_set1_ps(5.00000000e-01f));
    p = _mm256_fmadd_ps(p, r, _mm256_set1_ps(1.00000000e+00f));
    p = _mm256_fmadd_ps(p, r, _mm256_set1_ps(1.00000000e+00f));
    __m256i ki = _mm256_cvtps_epi32(kf);
    ki = _mm256_add_epi32(ki, _mm256_set1_epi32(127));
    ki = _mm256_slli_epi32(ki, 23);
    return _mm256_mul_ps(p, _mm256_castsi256_ps(ki));
}
static inline __m256 tkv_erf_ps(__m256 v) {
    const __m256 SQRT2INV = _mm256_set1_ps(0.70710678118654752f);
    __m256 arg = _mm256_mul_ps(v, SQRT2INV);
    __m256 sg  = _mm256_blendv_ps(_mm256_set1_ps(1.0f), _mm256_set1_ps(-1.0f),
                                  _mm256_cmp_ps(arg, _mm256_setzero_ps(), _CMP_LT_OQ));
    __m256 ax  = _mm256_andnot_ps(_mm256_set1_ps(-0.0f), arg);
    ax = _mm256_max_ps(ax, _mm256_setzero_ps());
    __m256 t   = _mm256_div_ps(_mm256_set1_ps(1.0f),
                               _mm256_fmadd_ps(_mm256_set1_ps(0.3275911f), ax, _mm256_set1_ps(1.0f)));
    __m256 p   = _mm256_set1_ps(1.061405429f);
    p = _mm256_fmadd_ps(p, t, _mm256_set1_ps(-1.453152027f));
    p = _mm256_fmadd_ps(p, t, _mm256_set1_ps(1.421413741f));
    p = _mm256_fmadd_ps(p, t, _mm256_set1_ps(-0.284496736f));
    p = _mm256_fmadd_ps(p, t, _mm256_set1_ps(0.254829592f));
    p = _mm256_fmadd_ps(p, t, _mm256_setzero_ps());
    __m256 negax2 = _mm256_max_ps(_mm256_mul_ps(_mm256_mul_ps(ax, ax), _mm256_set1_ps(-1.0f)),
                                  _mm256_set1_ps(-90.0f));
    __m256 e  = tkv_exp_ps(negax2);
    __m256 erf = _mm256_fnmadd_ps(e, p, _mm256_set1_ps(1.0f));
    return _mm256_mul_ps(sg, erf);
}
static inline __m256 tkv_gelu_bwd_one(__m256 v) {
    __m256 erf = tkv_erf_ps(v);
    __m256 d   = _mm256_mul_ps(_mm256_add_ps(_mm256_set1_ps(1.0f), erf),
                               _mm256_set1_ps(0.5f));
    __m256 neg = _mm256_max_ps(_mm256_mul_ps(_mm256_mul_ps(v, v), _mm256_set1_ps(-0.5f)),
                               _mm256_set1_ps(-90.0f));
    __m256 e   = tkv_exp_ps(neg);
    d = _mm256_fmadd_ps(_mm256_mul_ps(v, _mm256_set1_ps(0.3989422804014327f)), e, d);
    return d;
}
static inline float gelu_b_scalar(float v) {
    float arg = (float)((double)v / 1.4142135623730951);
    double sg = 1.0; if (arg < 0.0f) sg = -1.0;
    float ax = fabsf(arg);
    double t = 1.0 / (1.0 + 0.3275911 * (double)ax);
    double poly = 1.061405429;
    poly = poly * t + -1.453152027; poly = poly * t + 1.421413741;
    poly = poly * t + -0.284496736; poly = poly * t + 0.254829592;
    poly = poly * t;
    float e = (float)exp((double)((-ax) * ax));
    float ef = (float)(sg * (1.0 - (double)e * poly));
    double d = 0.5 * (1.0 + (double)ef)
             + ((double)v * 0.3989422804014327) * exp(((-0.5) * (double)v) * (double)v);
    return (float)d;
}
static inline void tkv_gelu_bv(const float* x, const float* gy, float* out, long long lo, long long hi) {
    long long i = lo;
    for (; i + 8 <= hi; i += 8) {
        __m256 xv = _mm256_loadu_ps(x + i);
        __m256 gv = _mm256_loadu_ps(gy + i);
        __m256 bv = tkv_gelu_bwd_one(xv);
        _mm256_storeu_ps(out + (i - lo), _mm256_mul_ps(bv, gv));
    }
    for (; i < hi; i++) out[i - lo] = (float)((double)gy[i] * gelu_b_scalar(x[i]));
}
"""
txt=txt.replace('#include <math.h>', '#include <math.h>\n'+avx)
open(cc_path,'w').write(txt)
print('AVX2 BWD helper injected =', 'tkv_gelu_bv' in open(cc_path).read())
p2=subprocess.run('gcc -O2 -mavx2 -mfma -shared -fPIC '+cc+' -o '+so+' -lm -lpthread -ldl', shell=True, capture_output=True, text=True, timeout=600)
print('tkv C build rc=',p2.returncode)
if p2.returncode!=0: print(p2.stderr[:2000])

In [ ]:
# 3/4 - bitwise compare GeluErfBwd
import ctypes, numpy as np
native=ctypes.CDLL('/content/TVmain/Release/tools/simd_native/tkv_simd_native.so')
args=[ctypes.POINTER(ctypes.c_float),ctypes.c_int,ctypes.POINTER(ctypes.c_float),ctypes.c_int,ctypes.POINTER(ctypes.c_float),ctypes.c_int,ctypes.c_longlong,ctypes.c_longlong]
for nm in ['native','tkv']:
    lib=native if nm=='native' else ctypes.CDLL('/content/TVcpu/Release/cuda_backend/examples/gelu_bwd_cpu.so')
    fn=lib.GeluErfBwd; fn.argtypes=args; fn.restype=ctypes.c_int
    if nm=='native': nf=fn
    else: tf=fn
rng=np.random.default_rng(7)
for lo,hi in [(0,64),(5,64),(0,1000),(3,777)]:
    x=rng.standard_normal(hi+4).astype(np.float32); gy=rng.standard_normal(hi+4).astype(np.float32)
    a=np.zeros(hi-lo,np.float32); c=np.zeros(hi-lo,np.float32)
    nf(x.ctypes.data_as(ctypes.POINTER(ctypes.c_float)),x.size,gy.ctypes.data_as(ctypes.POINTER(ctypes.c_float)),gy.size,a.ctypes.data_as(ctypes.POINTER(ctypes.c_float)),a.size,lo,hi)
    tf(x.ctypes.data_as(ctypes.POINTER(ctypes.c_float)),x.size,gy.ctypes.data_as(ctypes.POINTER(ctypes.c_float)),gy.size,c.ctypes.data_as(ctypes.POINTER(ctypes.c_float)),c.size,lo,hi)
    d=np.count_nonzero(a.view(np.uint32)!=c.view(np.uint32))
    print('lo=%d hi=%d bitwise=%s diff=%d/%d maxabs=%.3g'%(lo,hi,d==0,d,a.size,np.abs(a-c).max()))

In [ ]:
# 4/4 - summary
print('If all bitwise=True -> BiasAddRows can be replaced by .tkv-generated C.')